# Stock Price Forecast Benchmark

This notebook benchmarks **7 time-series forecasting models** on historical stock data.

| Setting | Value |
|---|---|
| Tickers | ^GSPC (S&P 500), KO (Coca-Cola), IBM |
| Training period | 1962 – 1990 |
| Test period | 1991 – 2000 |
| Target | Daily closing price |

**Models compared:**
1. ARIMA — classical statistical baseline
2. Prophet — Meta's decomposable trend + seasonality model
3. XGBoost — gradient-boosted trees on lag features
4. LightGBM — fast histogram-based gradient boosting
5. LSTM — long short-term memory recurrent network
6. GRU — gated recurrent unit (lighter LSTM variant)
7. Transformer — self-attention encoder

**Evaluation metrics:** MAE, RMSE, MAPE, Directional Accuracy (DA)

> GitHub repo: https://github.com/hurjun/stock-forecast-benchmark

In [ ]:
# Install all required packages.
# These are already available on Kaggle, but pip install ensures correct versions.
!pip install -q yfinance prophet lightgbm xgboost statsmodels

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import logging
import warnings

# ── Numerical / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Data download ─────────────────────────────────────────────────────────────
import yfinance as yf

# ── Statistical models ────────────────────────────────────────────────────────
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

# ── Gradient-boosting models ──────────────────────────────────────────────────
import xgboost as xgb
import lightgbm as lgb

# ── Deep learning ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress verbose output from Prophet / Stan / LightGBM
warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

# Fix all random seeds so results are reproducible
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print('All imports successful.')

## 1. Configuration
All hyperparameters are defined here — nothing is hardcoded inside the model classes.

In [ ]:
# ── Experiment configuration ───────────────────────────────────────────────────
CFG = {
    'data': {
        # Tickers chosen for their long data history on Yahoo Finance
        'tickers':     ['^GSPC', 'KO', 'IBM'],
        'train_start': '1962-01-01',
        'train_end':   '1990-12-31',
        'test_start':  '1991-01-01',
        'test_end':    '2000-12-31',
        'target_col':  'Close',
    },
    'features': {
        'lags':            30,          # Use the past 30 days as lag features
        'rolling_windows': [7, 14, 30], # Rolling mean + std over these windows
    },
    'models': {
        'arima':       {'order': [5, 1, 0]},
        'prophet':     {'weekly_seasonality': True, 'yearly_seasonality': True},
        'xgboost':     {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 5},
        'lightgbm':    {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 5},
        'lstm':        {'hidden_size': 128, 'num_layers': 2, 'seq_len': 60,
                        'epochs': 50, 'batch_size': 32, 'lr': 0.001},
        'gru':         {'hidden_size': 128, 'num_layers': 2, 'seq_len': 60,
                        'epochs': 50, 'batch_size': 32, 'lr': 0.001},
        'transformer': {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'seq_len': 60,
                        'epochs': 50, 'batch_size': 32, 'lr': 0.001},
    },
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Data Loading & Feature Engineering

In [ ]:
def download_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Download adjusted closing prices for a single ticker via yfinance."""
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)

    # Newer versions of yfinance return a MultiIndex column — flatten it
    if isinstance(df.columns, pd.MultiIndex):
        df = df.droplevel(1, axis=1)

    # Remove timezone information so date comparisons work cleanly
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)

    return df


def add_features(df: pd.DataFrame, lags: int, rolling_windows: list) -> pd.DataFrame:
    """
    Append lag and rolling-window features to the DataFrame.

    All features are shifted by 1 step (.shift(1)) so that the feature
    at row i only contains information from rows *before* i.
    This prevents look-ahead bias (data leakage).
    """
    df = df.copy()
    target = df['Close']

    # lag_k[i] = Close[i-k]  (the price k days ago)
    for lag in range(1, lags + 1):
        df[f'lag_{lag}'] = target.shift(lag)

    # Rolling statistics over the past w days (shifted to avoid leakage)
    for w in rolling_windows:
        df[f'rolling_mean_{w}'] = target.rolling(w).mean().shift(1)
        df[f'rolling_std_{w}']  = target.rolling(w).std().shift(1)

    # Drop any rows that still contain NaN (e.g. the first 30 rows)
    return df.dropna()


def load_ticker(ticker: str, cfg: dict):
    """Return (train_df, test_df) for a single ticker with features pre-computed."""
    d = cfg['data']
    f = cfg['features']

    # Download the full range (train + test) so lag features for the
    # first test rows correctly reference the end of the training period
    raw = download_data(ticker, d['train_start'], d['test_end'])
    df  = add_features(raw[[d['target_col']]], f['lags'], f['rolling_windows'])

    train = df.loc[d['train_start'] : d['train_end']]
    test  = df.loc[d['test_start']  : d['test_end']]
    print(f'  {ticker}: train={len(train)} rows, test={len(test)} rows')
    return train, test


# ── Load data for all tickers ──────────────────────────────────────────────────
print('Loading data...')
ticker_data = {}  # {ticker: (train_df, test_df)}
for t in CFG['data']['tickers']:
    ticker_data[t] = load_ticker(t, CFG)

print('Done.')

## 3. Model Definitions
Each model class exposes the same two-method interface:
- `fit(train_df)` — train on the training DataFrame
- `predict(n_steps)` — return a NumPy array of exactly `n_steps` future price predictions

In [ ]:
# ── 3a. ARIMA ─────────────────────────────────────────────────────────────────

class ARIMAForecaster:
    """ARIMA(p, d, q) statistical forecaster.

    p=5 autoregressive lags, d=1 difference (removes unit root / trend),
    q=0 moving-average terms.
    """
    name = 'ARIMA'

    def __init__(self, config: dict):
        self.order = tuple(config['order'])  # e.g. (5, 1, 0)

    def fit(self, train: pd.DataFrame):
        # ARIMA only needs the univariate Close series
        series = train['Close'].astype(float)
        result = ARIMA(series, order=self.order).fit()
        self._result = result
        print(f'  ARIMA fitted (AIC={result.aic:.1f})')

    def predict(self, n_steps: int) -> np.ndarray:
        # Forecast n_steps ahead from the end of the training series
        return self._result.forecast(steps=n_steps).to_numpy()

In [ ]:
# ── 3b. Prophet ───────────────────────────────────────────────────────────────

class ProphetForecaster:
    """Meta's Prophet model with trend + seasonality decomposition."""
    name = 'Prophet'

    def __init__(self, config: dict):
        self.config = config

    def fit(self, train: pd.DataFrame):
        # Prophet expects a DataFrame with columns 'ds' (date) and 'y' (target)
        df = (
            train[['Close']]
            .reset_index()
            .rename(columns={train.index.name or 'Date': 'ds', 'Close': 'y'})
        )
        df['ds'] = pd.to_datetime(df['ds'])

        self._model = Prophet(
            weekly_seasonality=self.config['weekly_seasonality'],
            yearly_seasonality=self.config['yearly_seasonality'],
        )
        self._model.fit(df)
        print('  Prophet fitted')

    def predict(self, n_steps: int) -> np.ndarray:
        # make_future_dataframe creates a schedule of n_steps business days
        # beyond the last training date
        future   = self._model.make_future_dataframe(periods=n_steps, freq='B')
        forecast = self._model.predict(future)
        # The DataFrame includes training dates, so we take only the last n_steps rows
        return forecast['yhat'].values[-n_steps:]

In [ ]:
# ── 3c. Shared helper for ML models (XGBoost & LightGBM) ─────────────────────

def _feature_cols(df: pd.DataFrame) -> list:
    """Return column names that are lag or rolling features."""
    return [c for c in df.columns if c.startswith(('lag_', 'rolling_'))]


def _build_features(history: list, lags: int, roll_wins: list) -> list:
    """
    Reconstruct the same feature vector used during training from a
    running history list.  Called at each step of recursive prediction.

    Feature order must match what was produced by add_features():
      lag_1, lag_2, ..., lag_lags,
      rolling_mean_w1, rolling_std_w1, rolling_mean_w2, ...
    """
    feats = []
    # Lag features: history[-1] is the most recent known value
    for lag in range(1, lags + 1):
        feats.append(history[-lag])
    # Rolling statistics computed from the tail of history
    for w in roll_wins:
        window = np.array(history[-w:])
        feats.append(float(window.mean()))
        feats.append(float(window.std()) if len(window) > 1 else 0.0)
    return feats


# ── 3d. XGBoost ───────────────────────────────────────────────────────────────

class XGBoostForecaster:
    """XGBoost regression model with recursive multi-step prediction."""
    name = 'XGBoost'

    def __init__(self, config: dict, feat_cfg: dict):
        self.config   = config
        self.lags     = feat_cfg['lags']
        self.roll_wins = feat_cfg['rolling_windows']

    def fit(self, train: pd.DataFrame):
        cols = _feature_cols(train)
        X = train[cols].values
        y = train['Close'].values.astype(float)

        self._model = xgb.XGBRegressor(
            n_estimators  = self.config['n_estimators'],
            learning_rate = self.config['learning_rate'],
            max_depth     = self.config['max_depth'],
            random_state  = SEED,
            verbosity     = 0,
        )
        self._model.fit(X, y)

        # Keep enough tail history to compute all features during prediction
        lookback = max(self.lags, max(self.roll_wins))
        self._history = list(train['Close'].values[-lookback:].astype(float))
        print(f'  XGBoost fitted')

    def predict(self, n_steps: int) -> np.ndarray:
        # Recursive strategy: predict one step → append prediction → repeat
        history = list(self._history)
        preds = []
        for _ in range(n_steps):
            feat = _build_features(history, self.lags, self.roll_wins)
            val  = float(self._model.predict(np.array([feat]))[0])
            preds.append(val)
            history.append(val)  # use prediction as the next lag
        return np.array(preds)

In [ ]:
# ── 3e. LightGBM ──────────────────────────────────────────────────────────────

class LightGBMForecaster:
    """LightGBM regression model — same interface and feature set as XGBoost."""
    name = 'LightGBM'

    def __init__(self, config: dict, feat_cfg: dict):
        self.config    = config
        self.lags      = feat_cfg['lags']
        self.roll_wins = feat_cfg['rolling_windows']

    def fit(self, train: pd.DataFrame):
        cols = _feature_cols(train)
        X = train[cols].values
        y = train['Close'].values.astype(float)

        self._model = lgb.LGBMRegressor(
            n_estimators  = self.config['n_estimators'],
            learning_rate = self.config['learning_rate'],
            max_depth     = self.config['max_depth'],
            random_state  = SEED,
            verbosity     = -1,  # suppress LightGBM console output
        )
        self._model.fit(X, y)

        lookback = max(self.lags, max(self.roll_wins))
        self._history = list(train['Close'].values[-lookback:].astype(float))
        print('  LightGBM fitted')

    def predict(self, n_steps: int) -> np.ndarray:
        history = list(self._history)
        preds = []
        for _ in range(n_steps):
            feat = _build_features(history, self.lags, self.roll_wins)
            val  = float(self._model.predict(np.array([feat]))[0])
            preds.append(val)
            history.append(val)
        return np.array(preds)

In [ ]:
# ── 3f. Shared helpers for PyTorch models ────────────────────────────────────

def _make_sequences(series: np.ndarray, seq_len: int):
    """Slide a window of length seq_len over the series to create (X, y) pairs."""
    X = np.array([series[i : i + seq_len] for i in range(len(series) - seq_len)])
    y = series[seq_len:]
    return X, y


def _train_torch_model(model, loader, epochs: int, lr: float, device):
    """Standard PyTorch training loop: Adam optimizer + MSE loss."""
    model.train()
    opt     = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()          # clear gradients from previous step
            loss = loss_fn(model(xb), yb)
            loss.backward()          # backpropagate
            opt.step()               # update weights


def _recursive_predict(model, last_seq: list, seq_len: int,
                        n_steps: int, mean: float, std: float, device) -> np.ndarray:
    """Auto-regressively predict n_steps ahead in normalised space, then denormalise."""
    model.eval()
    seq  = list(last_seq)
    preds = []

    with torch.no_grad():
        for _ in range(n_steps):
            # Build a (1, seq_len, 1) tensor from the most recent seq_len values
            x = (
                torch.tensor(seq[-seq_len:], dtype=torch.float32)
                .unsqueeze(0)
                .unsqueeze(-1)
                .to(device)
            )
            val = model(x).item()    # predicted normalised value
            preds.append(val)
            seq.append(val)          # feed prediction back as the next input

    # Reverse the z-score normalisation applied before training
    return np.array(preds) * std + mean

In [ ]:
# ── 3g. LSTM ──────────────────────────────────────────────────────────────────

class _LSTMNet(nn.Module):
    """Two-layer LSTM followed by a linear output layer."""
    def __init__(self, hidden_size: int, num_layers: int):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden_size, num_layers, batch_first=True)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, 1)
        out, _ = self.lstm(x)
        # Use the hidden state at the last time step for prediction
        return self.fc(out[:, -1, :]).squeeze(-1)


class LSTMForecaster:
    name = 'LSTM'

    def __init__(self, config: dict):
        self.cfg = config

    def fit(self, train: pd.DataFrame):
        series = train['Close'].values.astype(np.float32)

        # Z-score normalisation: helps gradient-based optimisation converge
        self._mean, self._std = series.mean(), series.std()
        norm = (series - self._mean) / self._std

        X, y = _make_sequences(norm, self.cfg['seq_len'])
        loader = DataLoader(
            TensorDataset(torch.tensor(X).unsqueeze(-1), torch.tensor(y)),
            batch_size=self.cfg['batch_size'], shuffle=True,
        )

        self._model = _LSTMNet(self.cfg['hidden_size'], self.cfg['num_layers']).to(DEVICE)
        _train_torch_model(self._model, loader, self.cfg['epochs'], self.cfg['lr'], DEVICE)
        self._last_seq = list(norm[-self.cfg['seq_len']:])
        print('  LSTM fitted')

    def predict(self, n_steps: int) -> np.ndarray:
        return _recursive_predict(
            self._model, self._last_seq, self.cfg['seq_len'],
            n_steps, self._mean, self._std, DEVICE
        )

In [ ]:
# ── 3h. GRU ───────────────────────────────────────────────────────────────────

class _GRUNet(nn.Module):
    """Two-layer GRU followed by a linear output layer."""
    def __init__(self, hidden_size: int, num_layers: int):
        super().__init__()
        self.gru = nn.GRU(1, hidden_size, num_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


class GRUForecaster:
    name = 'GRU'

    def __init__(self, config: dict):
        self.cfg = config

    def fit(self, train: pd.DataFrame):
        series = train['Close'].values.astype(np.float32)
        self._mean, self._std = series.mean(), series.std()
        norm = (series - self._mean) / self._std

        X, y = _make_sequences(norm, self.cfg['seq_len'])
        loader = DataLoader(
            TensorDataset(torch.tensor(X).unsqueeze(-1), torch.tensor(y)),
            batch_size=self.cfg['batch_size'], shuffle=True,
        )

        self._model = _GRUNet(self.cfg['hidden_size'], self.cfg['num_layers']).to(DEVICE)
        _train_torch_model(self._model, loader, self.cfg['epochs'], self.cfg['lr'], DEVICE)
        self._last_seq = list(norm[-self.cfg['seq_len']:])
        print('  GRU fitted')

    def predict(self, n_steps: int) -> np.ndarray:
        return _recursive_predict(
            self._model, self._last_seq, self.cfg['seq_len'],
            n_steps, self._mean, self._std, DEVICE
        )

In [ ]:
# ── 3i. Transformer ───────────────────────────────────────────────────────────

class _TransformerNet(nn.Module):
    """Transformer encoder for univariate sequence forecasting."""
    def __init__(self, d_model: int, nhead: int, num_layers: int):
        super().__init__()
        # Project scalar input to d_model dimensions before feeding to encoder
        self.input_proj = nn.Linear(1, d_model)
        encoder_layer   = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True,
            dim_feedforward=d_model * 4,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc      = nn.Linear(d_model, 1)

    def forward(self, x):
        x   = self.input_proj(x)   # (batch, seq, 1) → (batch, seq, d_model)
        out = self.encoder(x)      # self-attention over sequence
        return self.fc(out[:, -1, :]).squeeze(-1)  # predict from last position


class TransformerForecaster:
    name = 'Transformer'

    def __init__(self, config: dict):
        self.cfg = config

    def fit(self, train: pd.DataFrame):
        series = train['Close'].values.astype(np.float32)
        self._mean, self._std = series.mean(), series.std()
        norm = (series - self._mean) / self._std

        X, y = _make_sequences(norm, self.cfg['seq_len'])
        loader = DataLoader(
            TensorDataset(torch.tensor(X).unsqueeze(-1), torch.tensor(y)),
            batch_size=self.cfg['batch_size'], shuffle=True,
        )

        self._model = _TransformerNet(
            self.cfg['d_model'], self.cfg['nhead'], self.cfg['num_layers']
        ).to(DEVICE)
        _train_torch_model(self._model, loader, self.cfg['epochs'], self.cfg['lr'], DEVICE)
        self._last_seq = list(norm[-self.cfg['seq_len']:])
        print('  Transformer fitted')

    def predict(self, n_steps: int) -> np.ndarray:
        return _recursive_predict(
            self._model, self._last_seq, self.cfg['seq_len'],
            n_steps, self._mean, self._std, DEVICE
        )

## 4. Evaluation Metrics

In [ ]:
def mae(y_true, y_pred):
    """Mean Absolute Error — average absolute dollar deviation."""
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    """Root Mean Squared Error — penalises large errors more heavily than MAE."""
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error (%) — scale-independent, skips zero actuals."""
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

def directional_accuracy(y_true, y_pred):
    """Fraction of days where the model correctly predicts up vs. down movement."""
    return float(np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))) * 100)

def compute_all(y_true, y_pred):
    return {
        'MAE':  mae(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'MAPE': mape(y_true, y_pred),
        'DA':   directional_accuracy(y_true, y_pred),
    }

## 5. Run the Full Benchmark Pipeline

In [ ]:
def align(arr: np.ndarray, n: int) -> np.ndarray:
    """Ensure the prediction array is exactly n elements (trim or edge-pad)."""
    if len(arr) > n:  return arr[:n]
    if len(arr) < n:  return np.pad(arr, (0, n - len(arr)), mode='edge')
    return arr


def build_models(cfg):
    """Instantiate all 7 forecasters from config."""
    feat = cfg['features']
    m    = cfg['models']
    return [
        ARIMAForecaster(m['arima']),
        ProphetForecaster(m['prophet']),
        XGBoostForecaster(m['xgboost'], feat),
        LightGBMForecaster(m['lightgbm'], feat),
        LSTMForecaster(m['lstm']),
        GRUForecaster(m['gru']),
        TransformerForecaster(m['transformer']),
    ]


# ── Main loop ─────────────────────────────────────────────────────────────────
results       = []   # list of {model, ticker, MAE, RMSE, MAPE, DA}
all_forecasts = {}   # {ticker: {model_name: np.ndarray}}
all_actuals   = {}   # {ticker: np.ndarray}
all_dates     = {}   # {ticker: DatetimeIndex}

for ticker, (train_df, test_df) in ticker_data.items():
    print(f'\n=== {ticker} ===')
    actual  = test_df['Close'].values.astype(float)
    n_steps = len(test_df)

    all_actuals[ticker]   = actual
    all_dates[ticker]     = test_df.index
    all_forecasts[ticker] = {}

    for model in build_models(CFG):
        try:
            print(f'  Training {model.name}...')
            model.fit(train_df)

            preds   = align(model.predict(n_steps), n_steps)
            metrics = compute_all(actual, preds)

            results.append({'model': model.name, 'ticker': ticker, **metrics})
            all_forecasts[ticker][model.name] = preds

            print(f'    RMSE={metrics["RMSE"]:.4f}  DA={metrics["DA"]:.1f}%')

        except Exception as e:
            # Log and continue — one failing model should not stop the rest
            print(f'  [{model.name}] FAILED: {e}')

print('\nAll models trained.')

## 6. Leaderboard

In [ ]:
# Aggregate per-ticker results by averaging metrics across all tickers
df_results  = pd.DataFrame(results)
leaderboard = (
    df_results
    .groupby('model')[['MAE', 'RMSE', 'MAPE', 'DA']]
    .mean()
    .reset_index()
    .sort_values('RMSE')  # rank by RMSE (lower is better)
    .reset_index(drop=True)
)
leaderboard.insert(0, 'Rank', range(1, len(leaderboard) + 1))

print('\n=== Leaderboard (averaged across tickers) ===')
leaderboard.style.format({'MAE': '{:.4f}', 'RMSE': '{:.4f}', 'MAPE': '{:.2f}', 'DA': '{:.1f}'})

## 7. Visualisations

In [ ]:
# ── Forecast comparison for the primary ticker ─────────────────────────────────
primary = CFG['data']['tickers'][0]  # ^GSPC used as the showcase ticker

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(all_dates[primary], all_actuals[primary],
        label='Actual', color='black', linewidth=2)

palette = plt.get_cmap('tab10').colors
for i, (model_name, preds) in enumerate(all_forecasts[primary].items()):
    ax.plot(all_dates[primary], preds,
            label=model_name, color=palette[i % len(palette)],
            linewidth=1.2, alpha=0.85)

ax.set_title(f'Forecast Comparison — {primary} (Test Period: 1991–2000)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Close Price (USD)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Metrics bar chart ─────────────────────────────────────────────────────────
metrics_to_plot = ['MAE', 'RMSE', 'MAPE']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric in zip(axes, metrics_to_plot):
    sns.barplot(data=leaderboard, x='model', y=metric, ax=ax, palette='tab10')
    ax.set_title(metric, fontsize=12)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=40)

fig.suptitle('Model Comparison — Average Across Tickers (1991–2000)', y=1.03, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Directional Accuracy (higher is better) ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=leaderboard, x='model', y='DA', ax=ax, palette='tab10')
ax.axhline(50, color='red', linestyle='--', linewidth=1, label='Random baseline (50%)')
ax.set_title('Directional Accuracy (%) — higher is better', fontsize=12)
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=40)
ax.legend()
plt.tight_layout()
plt.show()